- cpu vs. io
    - Python 的 asyncio 适合 I/O 密集型（爬虫、Web服务、数据库读写）。
    - 不适合 CPU 密集型（视频解码、复杂的数学计算）。如果非要在 async 函数里算圆周率，整个程序都会卡死，因为单线程的 CPU 被占用了，Loop 没机会去调度别的任务。

### event loop

```python
import asyncio
import time

async def toast_bread():
    print("开始烤面包...")
    await asyncio.sleep(3) 
    print("面包烤好了")

async def brew_coffee():
    print("开始冲咖啡...")
    await asyncio.sleep(1)
    print("咖啡冲好了")

async def main():
    start_time = time.time()  # 记录开始时间
    
    # 并发运行
    await asyncio.gather(toast_bread(), brew_coffee())
    
    end_time = time.time()    # 记录结束时间
    print(f"------------\n总耗时: {end_time - start_time:.2f} 秒")

if __name__ == "__main__":
    asyncio.run(main())
```
- log
    - 开始烤面包... (遇到 await，切换)
    - 开始冲咖啡... (遇到 await，切换)
    - 咖啡冲好了 (1秒后，咖啡任务先完成)
    - 面包烤好了 (3秒后，面包任务完成)
- `asyncio.run(main())`—— Loop 的出生与消亡
    - Python 自动创建了一个新的 Event Loop 对象（你可以理解为“招聘了一位调度员”）。
    - 它把 main() 这个协程扔进 Loop 里去运行。
    - Loop 开始不断循环，直到 main() 完成。
    - 最后，它自动关闭并清理 Loop。
- 让出控制权：await —— Loop 介入调度的时刻
- 任务注册：asyncio.gather(...) —— 往 Loop 里塞任务

### asyncio（协程）vs. 线程

- `toast_bread` 和 `brew_coffee` 运行在同一个线程（Thread）里，甚至有着完全相同的线程 ID。
- Python `asyncio`（协程）最神奇、也最容易让人困惑的地方：它是在单线程里实现了“并发”。

```python
import asyncio
import threading  # 引入线程模块用来查看ID
import time

async def toast_bread():
    # 打印当前线程 ID
    print(f"[面包] 正在运行，线程 ID: {threading.get_ident()}")
    print("开始烤面包...")
    await asyncio.sleep(3) 
    print(f"[面包] 烤好了，线程 ID: {threading.get_ident()}")

async def brew_coffee():
    # 打印当前线程 ID
    print(f"[咖啡] 正在运行，线程 ID: {threading.get_ident()}")
    print("开始冲咖啡...")
    await asyncio.sleep(1)
    print(f"[咖啡] 冲好了，线程 ID: {threading.get_ident()}")

async def main():
    print(f"[主程序] 正在运行，线程 ID: {threading.get_ident()}")
    await asyncio.gather(toast_bread(), brew_coffee())

if __name__ == "__main__":
    asyncio.run(main())
```

### 耗时分析

- Python 自带的 requests 库是同步的（不支持 await），在异步编程中我们通常用 httpx 或 aiohttp。
- `await client.get(...)`
    - 发送数据：Python 调用操作系统底层的 socket，把“查询请求”的数据包扔到了网卡上。
    - 交出权限：网卡把数据顺着网线发出去了。此时，Python 知道对方服务器响应需要时间（光速传播+服务器处理）。
- 注意 await 本身不是异步的，“去后台跑，我继续往下走”，而是“暂停在这里，等结果回来”。（await 这个词在英语里就是“等”）
    - `resp = await client.get(url)`
        - 对内（当前函数）：它是一个红灯。 它强制当前这个 fetch_price 函数暂停执行。只要网络请求没回来，程序就死死卡在这一行，绝对不会往下运行去执行 print。
        - 对外（Event Loop）：它是一个绿灯。 它虽然让自己停下了，但它同时告诉 Event Loop：“老大，我要等数据，暂时没法干活了。你切走吧，把 CPU 让给别的任务（比如去处理淘宝或拼多多的请求）。”
- await 的目的，就是为了让你用写同步代码（一步一步往下写）的逻辑，写出异步的程序。

```python
import asyncio
import httpx  # 需要 pip install httpx
import time

# 模拟真实的请求函数
async def fetch_price(platform_name, url, sleep_time):
    print(f"1. [发送请求] 正在连接 {platform_name}...")
    
    # -------------------------------------------------------------
    # 核心替换：这里不再是假的 sleep，而是真实的 HTTP 请求
    # 这里的 await 表示：请求发出去后，网络数据要在光缆里跑一会儿
    # CPU 不需要跟着跑，所以把控制权交还给 Loop
    # -------------------------------------------------------------
    async with httpx.AsyncClient() as client:
        # 为了演示效果，我们强制让这个真实请求“慢”一点 (模拟服务器响应慢)
        # 在真实世界里，这里就是 await client.get(url)
        resp = await client.get(f"https://httpbin.org/delay/{sleep_time}")
        
    print(f"2. [接收响应] 拿到 {platform_name} 的价格了！")
    return f"{platform_name}: 99元"

async def main():
    start = time.time()
    
    print("--- 开始比价 ---")
    # 同时向三家电商发出请求
    # Loop 会在发完第一个请求后，立刻发第二个，不用等第一个回来
    results = await asyncio.gather(
        fetch_price("京东", "jd.com", 5),   # 模拟延时3秒
        fetch_price("淘宝", "taobao.com", 3), # 模拟延时2秒
        fetch_price("拼多多", "pdd.com", 2)   # 模拟延时1秒
    )
    
    print("--- 比价结束 ---")
    for r in results:
        print(r)
        
    print(f"总耗时: {time.time() - start:.2f} 秒")

if __name__ == "__main__":
    asyncio.run(main())
```